<a href="https://colab.research.google.com/github/subhadeepm465/data_engineering_trng/blob/Dev/Pyspark_Session2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark findspark

In [ ]:

import pandas as pd
users = pd.read_csv('https://raw.githubusercontent.com/justmarkham/DAT8/master/data/u.user',
                      sep='|', index_col='user_id')

In [ ]:
import findspark
findspark.init()

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("PysparkPractice").getOrCreate()


In [ ]:
df = spark.createDataFrame(users)


In [ ]:
df.select("occupation").distinct().count()


21

**Windows Function**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Initialize a Spark session
spark = SparkSession.builder \
    .appName("WindowFunctionsExamples") \
    .getOrCreate()

# Sample data
data = [
    (24, "M", "technician", "85711"),
    (53, "F", "other", "94043"),
    (23, "M", "writer", "32067"),
    (24, "M", "technician", "43537"),
    (33, "F", "other", "15213"),
    (33, "F", "technician", "15216")
]

# Create DataFrame
columns = ["age", "gender", "occupation", "zip_code"]
df = spark.createDataFrame(data, columns)

# Show the data
df.show()


+---+------+----------+--------+
|age|gender|occupation|zip_code|
+---+------+----------+--------+
| 24|     M|technician|   85711|
| 53|     F|     other|   94043|
| 23|     M|    writer|   32067|
| 24|     M|technician|   43537|
| 33|     F|     other|   15213|
| 33|     F|technician|   15216|
+---+------+----------+--------+



In [ ]:
# Register DataFrame as a temporary SQL table/view
df.createOrReplaceTempView("users")


**1. ROW_NUMBER() - Assign Row Numbers Partitioned by Occupation**

In [ ]:
spark.sql("""
SELECT
    age,
    gender,
    occupation,
    ROW_NUMBER() OVER (partition by occupation order by age) AS row_num
FROM users
""").show()


+---+------+----------+-------+
|age|gender|occupation|row_num|
+---+------+----------+-------+
| 33|     F|     other|      1|
| 53|     F|     other|      2|
| 24|     M|technician|      1|
| 24|     M|technician|      2|
| 33|     F|technician|      3|
| 23|     M|    writer|      1|
+---+------+----------+-------+



In [ ]:
spark.sql("""
SELECT
    age,
    gender,
    occupation,
    ROW_NUMBER() OVER (PARTITION BY occupation ORDER BY age) AS row_num
FROM users
""").show()


+---+------+----------+-------+
|age|gender|occupation|row_num|
+---+------+----------+-------+
| 33|     F|     other|      1|
| 53|     F|     other|      2|
| 24|     M|technician|      1|
| 24|     M|technician|      2|
| 33|     F|technician|      3|
| 23|     M|    writer|      1|
+---+------+----------+-------+



In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Define window specification
window_spec = Window.partitionBy("occupation").orderBy("age")

# Add row number
df_with_row_num = df.withColumn("row_num", F.row_number().over(window_spec))
df_with_row_num.show()


In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

wd = Window.partitionBy('occupation').orderBy(col('age').desc())

df1 = df.withColumn("row_num",row_number().over(wd))
df1.show()

+---+------+----------+--------+-------+
|age|gender|occupation|zip_code|row_num|
+---+------+----------+--------+-------+
| 53|     F|     other|   94043|      1|
| 33|     F|     other|   15213|      2|
| 33|     F|technician|   15216|      1|
| 24|     M|technician|   85711|      2|
| 24|     M|technician|   43537|      3|
| 23|     M|    writer|   32067|      1|
+---+------+----------+--------+-------+



In [ ]:
df2 = df.withColumn("rank",rank().over(wd))
df2.show()

+---+------+----------+--------+----+
|age|gender|occupation|zip_code|rank|
+---+------+----------+--------+----+
| 53|     F|     other|   94043|   1|
| 33|     F|     other|   15213|   2|
| 33|     F|technician|   15216|   1|
| 24|     M|technician|   85711|   2|
| 24|     M|technician|   43537|   2|
| 23|     M|    writer|   32067|   1|
+---+------+----------+--------+----+



In [ ]:
wd = Window.partitionBy('occupation').orderBy(col('age').desc())
df2 = df.withColumn("lead",lead(col('age')).over(wd)).withColumn('lag',lag(col('age')).over(wd))
df2.show()

+---+------+----------+--------+----+----+
|age|gender|occupation|zip_code|lead| lag|
+---+------+----------+--------+----+----+
| 53|     F|     other|   94043|  33|NULL|
| 33|     F|     other|   15213|NULL|  53|
| 33|     F|technician|   15216|  24|NULL|
| 24|     M|technician|   85711|  24|  33|
| 24|     M|technician|   43537|NULL|  24|
| 23|     M|    writer|   32067|NULL|NULL|
+---+------+----------+--------+----+----+



In [ ]:
df.select(col('age'),col('gender')).show()
#df.select('age','gender').show()

+---+------+
|age|gender|
+---+------+
| 24|     M|
| 53|     F|
| 23|     M|
| 24|     M|
| 33|     F|
| 33|     F|
+---+------+



**2. RANK() - Rank Users Based on Age within Occupation**

In [ ]:
spark.sql("""
SELECT
    age,
    gender,
    occupation,
    RANK() OVER (PARTITION BY occupation ORDER BY age ) AS rank
FROM users
""").show()


+---+------+----------+----+
|age|gender|occupation|rank|
+---+------+----------+----+
| 33|     F|     other|   1|
| 53|     F|     other|   2|
| 24|     M|technician|   1|
| 24|     M|technician|   1|
| 33|     F|technician|   3|
| 23|     M|    writer|   1|
+---+------+----------+----+



In [ ]:
# Add rank
df_with_rank = df.withColumn("rank", F.rank().over(window_spec))
df_with_rank.show()


**3. DENSE_RANK() - Rank Users Based on Age within Occupation**

In [ ]:
spark.sql("""
SELECT
    age,
    gender,
    occupation,
    DENSE_RANK() OVER (PARTITION BY occupation ORDER BY age ) AS rank
FROM users
""").show()


+---+------+----------+----+
|age|gender|occupation|rank|
+---+------+----------+----+
| 33|     F|     other|   1|
| 53|     F|     other|   2|
| 24|     M|technician|   1|
| 24|     M|technician|   1|
| 33|     F|technician|   2|
| 23|     M|    writer|   1|
+---+------+----------+----+



In [ ]:
# Add rank
df_with_rank = df.withColumn("rank", F.dense_rank().over(window_spec))
df_with_rank.show()


**4. LEAD() - Show the Next User’s Age**

In [ ]:
spark.sql("""
SELECT
    age,
    gender,
    occupation,
    LEAD(age) OVER (PARTITION BY occupation ORDER BY age) AS next_age,
    LAG(age) OVER (PARTITION BY occupation ORDER BY age ) AS prev_age
FROM users
""").show()


+---+------+----------+--------+--------+
|age|gender|occupation|next_age|prev_age|
+---+------+----------+--------+--------+
| 33|     F|     other|      53|    NULL|
| 53|     F|     other|    NULL|      33|
| 24|     M|technician|      24|    NULL|
| 24|     M|technician|      33|      24|
| 33|     F|technician|    NULL|      24|
| 23|     M|    writer|    NULL|    NULL|
+---+------+----------+--------+--------+



In [ ]:
# Add next user's age
df_with_lead = df.withColumn("next_age", F.lead("age").over(window_spec))
df_with_lead.show()


**5. SUM() - Total Age by Occupation**

In [ ]:
spark.sql("""
SELECT
    age,
    gender,
    occupation,
    SUM(age) OVER (PARTITION BY occupation order by age ) AS total_age
FROM users
""").show()


+---+------+----------+---------+
|age|gender|occupation|total_age|
+---+------+----------+---------+
| 33|     F|     other|       33|
| 53|     F|     other|       86|
| 24|     M|technician|       48|
| 24|     M|technician|       48|
| 33|     F|technician|       81|
| 23|     M|    writer|       23|
+---+------+----------+---------+



In [ ]:
# Add total age within occupation
df_with_sum = df.withColumn("total_age", F.sum("age").over(window_spec))
df_with_sum.show()


**🔍 Basic Exploration**

Show the first 3 rows of the DataFrame.

Print the schema of the DataFrame.

List all column names.

Count the total number of records.

Show distinct occupations.

**🧹 Filtering & Conditions**

Filter users where gender is 'F'.

Filter users who are older than 30.

Filter users whose occupation is 'technician' and age > 23.

Find users with a zip code starting with '9'.

Find users who are either 'writer' or 'other'.

**✏️ Column Transformations**

Add a new column age_plus_5 which is age + 5.

Create a column is_senior (True if age >= 50).

Rename column zip_code to postal_code.

Drop the gender column.

Reorder columns as: occupation, age, gender, zip_code.

**📊 Aggregation & Grouping**

Count how many users are in each occupation.

Find the average age by gender.

Find the maximum and minimum age for each occupation.

Count users by zip_code.

Group by gender and occupation and count users.

**🔁 Window Functions (Medium Level)**

Assign a row number partitioned by occupation and ordered by age.

For each occupation, calculate the average age using a window function.

Add a cumulative sum of age ordered by age.

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Define window specification
window_spec = Window.partitionBy("occupation").orderBy("age")

# Add row number to the DataFrame
df_with_row_num = users.withColumn("row_number", row_number().over(window_spec))

df_with_row_num.show()


In [ ]:
from pyspark.sql.functions import avg

# Define window specification
window_spec = Window.partitionBy("occupation")

# Add average age column
df_with_avg_age = users.withColumn("avg_age", avg("age").over(window_spec))

df_with_avg_age.show()


In [ ]:
from pyspark.sql.functions import sum as spark_sum

# Define window specification
window_spec = Window.orderBy("age")

# Add cumulative sum of age column
df_with_cumulative_sum = users.withColumn("cumulative_age", spark_sum("age").over(window_spec))

df_with_cumulative_sum.show()
